# 2D Heliosphere Model


# Setup


## Imports


In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/helio_n_matplotlib")

cwd = Path.cwd().resolve()
project_root = None
for candidate in (cwd, *cwd.parents, Path("/home/smdc/helio-n")):
    if (candidate / "Library").exists() and (candidate / "Config").exists():
        project_root = candidate
        break
assert (
    project_root is not None
), "Could not locate the helio-n project root for notebook imports."
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
from ipywidgets import interact, widgets

from Library.ICME import build_icme_mask

from Library.SW.Ballistic import (
    init_accumulators,
    postprocess_max_field,
    prepare_seed_inputs,
    run_bulk_propagation,
)
from Library.SW.Config import (
    load_ballistic_spec,
    load_sw_runtime_spec,
)
from Library.SW.Constants import CARRINGTON_ROTATION_DAYS
from Library.SW.Coords import (
    build_grid_axes,
    build_transport_state,
    compute_rotation_state,
)
from Library.SW.Inputs import (
    DEFAULT_SQL_CONNECTION,
    DEFAULT_SQL_QUERY,
    build_ace_earth_swx_frame,
    build_model_input_series,
    load_ace_earth_frame,
    load_sw_input_frame,
)
from Library.SW.Visualization import (
    build_satellite_comparison_frame,
    export_polar_animation,
    export_solar_wind_plot,
    plot_polar_snapshot,
)
from Models.CH_SW_Correspondence.Shugay import load as load_shugay
from Models.CH_SW_Correspondence.Shugay_Slow_SW import load as load_shugay_slow_sw

## Global Parameters


In [ ]:
empirical = load_shugay()
slow_sw_empirical = load_shugay_slow_sw()
ballistic = load_ballistic_spec()
runtime = load_sw_runtime_spec()
slow_sw_patch = True

# Keep short aliases for the repeated physics terms used throughout the notebook.
superresolution_enabled = bool(ballistic["superresolution_enabled"])

time_step_minutes = (
    int(ballistic["superresolution_step_minutes"])
    if superresolution_enabled
    else int(ballistic["base_time_step_minutes"])
)
time_step_hours = float(time_step_minutes) / 60.0
time_freq = f"{int(time_step_minutes)}min"
phi_step_minutes = ballistic["phi_step_minutes"]

r0 = ballistic["r0"]
R_max = ballistic["r_max"]
r_step = ballistic["r_step"]
horizon_hours = ballistic["horizon_hours"]
memory_guard_enabled = ballistic["memory_guard_enabled"]
simulation_pad_days = ballistic["simulation_pad_days"]

dense_memory_budget_gb = runtime["dense_memory_budget_gb"]
max_seed_batch = runtime["max_seed_batch"]
post_chunk_t = runtime["post_chunk_t"]

print(
    "Shugay.py:",
    empirical.source_path,
    "| Ballistic.json:",
    ballistic["json_path"],
)
print(
    "superresolution_enabled:",
    superresolution_enabled,
    "| dt(min):",
    time_step_minutes,
    "| freq:",
    time_freq,
)
print("phi_step_minutes:", phi_step_minutes)
print(
    "dense_memory_budget_gb:",
    dense_memory_budget_gb,
    "| max_seed_batch:",
    max_seed_batch,
)

Shugay.py: /Users/aosh/Developer/helio-n/Models/CH_SW_Correspondence/Shugay.py | Ballistic.json: /Users/aosh/Developer/helio-n/Config/SW/Ballistic.json
superresolution_enabled: True | dt(min): 2 | freq: 2min
phi_step_minutes: 120
dense_memory_budget_gb: 12.0 | max_seed_batch: 256


## Data Loading And Input Speed Series


In [ ]:
start_dt = pd.Timestamp("2018-03-01 00:00:00")
end_dt = pd.Timestamp("2018-06-01 00:00:00")

input_source = "sql"
input_parquet_path = Path("Data/CH Area.parquet")
sql_query = DEFAULT_SQL_QUERY
sql_connection_kwargs = DEFAULT_SQL_CONNECTION.copy()

In [ ]:
df_sdo_sw_test = load_sw_input_frame(
    start_dt=start_dt,
    end_dt=end_dt,
    source=input_source,
    input_parquet_path=input_parquet_path,
    query=sql_query,
    connection_kwargs=sql_connection_kwargs,
)
print(
    "Loaded propagation input from:",
    input_parquet_path if input_source == "parquet" else "SQL query",
)
print("Rows in filtered input:", len(df_sdo_sw_test))
df_sdo_sw_test.head()

Loaded propagation input from: SQL query
Rows in filtered input: 1307


,dt,forecast_dt,forecast_sw_speed,ch_relative_area
0,2018-02-24 02:00:00,2018-03-01 01:41:03.166383,347.210000,0.0
1,2018-02-24 03:00:00,2018-03-01 04:24:35.451026,342.275000,0.0
2,2018-02-24 07:00:00,2018-03-01 12:23:12.583864,331.418826,0.0
3,2018-02-24 08:00:00,2018-03-01 13:28:15.752792,331.196384,0.0
4,2018-02-24 09:00:00,2018-03-01 14:33:19.329229,330.973942,0.0


## Empirical SW Speed Model


In [11]:
sample_time = pd.date_range("2017-01-01 00:00:00", periods=3, freq="1h")
sample_area = pd.Series([0.0, 0.33, 0.66], index=sample_time, name="ch_relative_area")
pd.DataFrame(
    {
        "ch_relative_area": sample_area,
        "v_empirical": empirical.v_from_area(sample_area, t=sample_time),
    }
)

,ch_relative_area,v_empirical
2017-01-01 00:00:00,0.00,300.000000
2017-01-01 01:00:00,0.33,392.550947
2017-01-01 02:00:00,0.66,440.281004


## Simulation Input Preparation


In [12]:
prepared_inputs = build_model_input_series(
    sdo_input_df=df_sdo_sw_test,
    empirical=empirical,
    superresolution_enabled=superresolution_enabled,
    time_freq=time_freq,
    simulation_pad_days=simulation_pad_days,
)

sdo_input_df = prepared_inputs["sdo_input_df"]
df_v = prepared_inputs["df_v"]
df_ch_area = prepared_inputs["df_ch_area"]
sim_start = prepared_inputs["sim_start"]
sim_end = prepared_inputs["sim_end"]

print("Rows in df_v:", len(df_v), "| cadence:", time_freq)
print("Rows in df_ch_area:", len(df_ch_area))
df_v.head()

Rows in df_v: 67021 | cadence: 2min
Rows in df_ch_area: 67021


,v
time,
2018-02-24 02:00:00,300.0
2018-02-24 02:02:00,300.0
2018-02-24 02:04:00,300.0
2018-02-24 02:06:00,300.0
2018-02-24 02:08:00,300.0


# Model Definition


## Rotation And Geometry Constants


In [13]:
rotation = compute_rotation_state(phi_step_minutes=phi_step_minutes)
cr_time = rotation.cr_time
omega = rotation.omega
phi_step = rotation.phi_step

print("omega * 3600:", omega * 3600)
print("phi_step_used (deg):", phi_step, "| from phi_step_minutes:", phi_step_minutes)
print(
    "time_step_used (h):",
    time_step_hours,
    "| freq:",
    time_freq,
    "| superresolution:",
    superresolution_enabled,
)
print("r0:", r0, "| R_max:", R_max, "| r_step:", r_step)

omega * 3600: 0.5499489048596852
phi_step_used (deg): 1.0998978097193703 | from phi_step_minutes: 120
time_step_used (h): 0.03333333333333333 | freq: 2min | superresolution: True
r0: 20 | R_max: 215 | r_step: 1.5


## Grid Construction (time, phi, R)


In [14]:
grid = build_grid_axes(
    sim_start=sim_start,
    sim_end=sim_end,
    time_freq=time_freq,
    phi_step=phi_step,
    r0=r0,
    r_max=R_max,
    r_step=r_step,
    dense_memory_budget_gb=dense_memory_budget_gb,
    memory_guard_enabled=memory_guard_enabled,
)

time_axis = grid.time_axis
phi_axis = grid.phi_axis
r_axis = grid.r_axis
n_cells = grid.n_cells
est_runtime_gb = grid.est_runtime_gb

print("Grid shape (t, phi, R):", (len(time_axis), len(phi_axis), len(r_axis)))
print(f"Estimated dense propagation memory: {est_runtime_gb:.2f} GB")

AssertionError: Estimated memory 19.82 GB exceeds budget 12.00 GB. Reduce date range, increase phi_step_minutes, or disable superresolution.

In [ ]:
transport = build_transport_state(
    time_axis=time_axis,
    phi_axis=phi_axis,
    rotation_state=rotation,
    horizon_hours=horizon_hours,
    time_step_hours=time_step_hours,
)

t0_ref = transport.t0_ref
horizon_steps = transport.horizon_steps
h_step_idx = transport.h_step_idx
r_kernel_scale = transport.r_kernel_scale
cr_steps = transport.cr_steps
phi_delay_steps = transport.phi_delay_steps
phi_delay_offsets = transport.phi_delay_offsets
phi_delay_alpha = transport.phi_delay_alpha

print("Initialized dense propagation grid.")
print("time range:", time_axis.min(), "->", time_axis.max())
print("phi bins:", len(phi_axis), "R bins:", len(r_axis))

valid_seed_mask = (df_v.index >= time_axis.min()) & (df_v.index <= time_axis.max())
df_v_run = df_v.loc[valid_seed_mask].copy()
print("Seeds in simulation range:", len(df_v_run), "of", len(df_v))

Initialized dense propagation grid.
time range: 2018-02-24 02:00:00 -> 2018-06-04 04:00:00
phi bins: 328 R bins: 131
Seeds in simulation range: 1307 of 1307


## Propagation Functions


In [ ]:
acc = init_accumulators(
    n_t=len(time_axis),
    n_p=len(phi_axis),
    n_r=len(r_axis),
)

V_accum_max = acc.V_accum_max
cr_flat = acc.cr_flat
dims = (len(time_axis), len(phi_axis), len(r_axis))

(
    seed_vals,
    v_prev,
    v_next,
    seed_t_idx,
    seed_cr_idx_arr,
    seed_r_idx,
) = prepare_seed_inputs(
    df_v_run=df_v_run,
    cr_steps=cr_steps,
    horizon_steps=horizon_steps,
    time_freq=time_freq,
    t0_ref=t0_ref,
    time_step_hours=time_step_hours,
    r_kernel_scale=r_kernel_scale,
    r0=r0,
    r_axis=r_axis,
)

# Experiments


## Bulk Propagation Across Time


In [ ]:
stats = run_bulk_propagation(
    seed_vals=seed_vals,
    v_prev=v_prev,
    v_next=v_next,
    seed_t_idx=seed_t_idx,
    seed_cr_idx_arr=seed_cr_idx_arr,
    seed_r_idx=seed_r_idx,
    h_step_idx=h_step_idx,
    phi_delay_offsets=phi_delay_offsets,
    phi_delay_alpha=phi_delay_alpha,
    n_t=len(time_axis),
    n_p=len(phi_axis),
    n_r=len(r_axis),
    V_accum_max=V_accum_max,
    cr_flat=cr_flat,
    max_seed_batch=max_seed_batch,
)

filled = stats.filled
total = stats.total
prop_seconds = stats.prop_seconds

print(
    "time step (h):",
    time_step_hours,
    "| bins per hour:",
    int(round(1.0 / time_step_hours)),
)
print(
    "Accumulated non-empty cells:",
    filled,
    "/",
    total,
    f"({100.0 * filled / total:.2f}%)",
)
print(f"Propagation runtime: {prop_seconds:.2f} s ({prop_seconds / 60.0:.2f} min)")
print(f"Seeds processed: {stats.seeds_processed}")

2D propagate:   0%|          | 0/6 [00:00<?, ?batch/s]

time step (h): 1.0 | bins per hour: 1
Accumulated non-empty cells: 59763411 / 103252104 (57.88%)
Propagation runtime: 1.22 s (0.02 min)
Seeds processed: 1307


## Post-Processing


In [ ]:
slow_sw_speed = empirical.slow_sw_speed(time_axis)
slow_sw_patch_speed = slow_sw_empirical.slow_sw_speed(time_axis)

post = postprocess_max_field(
    V_accum_max=V_accum_max,
    slow_sw_speed=slow_sw_speed,
    post_chunk_t=post_chunk_t,
)

V_grid = post.V_grid
slow_sw_pred_mask = post.max_slow_sw_pred_mask
post_vlims_raw = post.max_vlims_raw
post_fields = V_grid
post_vlims = post_vlims_raw

print("Predicted slow-SW cells (max mode):", int(slow_sw_pred_mask.sum()))
print("raw v min/max (max mode):", post_vlims_raw)

Post-processing:   0%|          | 0/19 [00:00<?, ?chunk/s]

Predicted slow-SW cells (max mode): 32002426
raw v min/max (max mode): (300.0, 573.6466674804688)


# Analysis


## Visualization


## Satellite Data


In [ ]:
stereo_parquet_path = Path(
    "Data/STEREO-A Plastic.parquet"
)
earth_radii_per_solar_radius = 109.0763707060096

stereo_a_df = pd.read_parquet(
    stereo_parquet_path,
    columns=[
        "V",
        "radialDistance",
        "heliographicLatitude",
        "heliographicLongitude",
    ],
).copy()
stereo_a_df.index = pd.to_datetime(stereo_a_df.index, utc=True).tz_convert(None)
stereo_a_df = stereo_a_df.loc[time_axis.min() : time_axis.max()]
stereo_a_df = stereo_a_df.rename(
    columns={
        "V": "v",
        "radialDistance": "r_target",
        "heliographicLatitude": "lat_hge",
        "heliographicLongitude": "phi_target",
    }
)
for column in stereo_a_df.columns:
    stereo_a_df[column] = pd.to_numeric(stereo_a_df[column], errors="coerce")
stereo_a_df["r_target"] = stereo_a_df["r_target"] / earth_radii_per_solar_radius
stereo_a_df = stereo_a_df.resample(time_freq).mean().reindex(time_axis)
stereo_a_df.attrs["sat"] = "stereo_a"
stereo_a_df.attrs["label"] = "STEREO-A"
stereo_a_df.attrs["coord_frame"] = "HGE"
stereo_a_df

,v,r_target,lat_hge,phi_target
2018-02-24 02:00:00,NaN,NaN,NaN,NaN
2018-02-24 03:00:00,NaN,NaN,NaN,NaN
2018-02-24 04:00:00,NaN,NaN,NaN,NaN
2018-02-24 05:00:00,NaN,NaN,NaN,NaN
2018-02-24 06:00:00,NaN,NaN,NaN,NaN
...,...,...,...,...
2018-06-04 00:00:00,292.449477,208.009702,-6.569948,-114.737466
2018-06-04 01:00:00,289.034716,208.009519,-6.572419,-114.733790
2018-06-04 02:00:00,287.498385,208.009346,-6.574757,-114.730306
2018-06-04 03:00:00,298.166621,208.009138,-6.577546,-114.726142


## WSA/ENLIL Predictions


In [ ]:
enlil_parquet_path = Path("Data/ENLIL 2018-02-01 2018-07-01.parquet")

# Each background run provides ~5 d of forecast past its issue date. We lock
# every prediction timestamp to a single run at a fixed lead, matching our
# CH-area-driven forecast's effective operational horizon. Daily issuance
# means at most one run satisfies the window per timestamp.
enlil_lead_days = 5.0
enlil_lead_tol = pd.Timedelta(hours=12)

_enlil_raw = pd.read_parquet(
    enlil_parquet_path,
    columns=["time", "run_id", "Earth_V1", "STEREO_A_V1"],
)
_enlil_raw["time"] = pd.to_datetime(_enlil_raw["time"], utc=True).dt.tz_convert(None)
_enlil_raw["issue_dt"] = pd.to_datetime(
    _enlil_raw["run_id"].str.extract(r"_(\d{8})_\d{4}$", expand=False),
    format="%Y%m%d",
)
_enlil_raw["lead"] = _enlil_raw["time"] - _enlil_raw["issue_dt"]

_target_lead = pd.Timedelta(days=enlil_lead_days)
_enlil_selected = _enlil_raw.loc[
    (_enlil_raw["lead"] >= _target_lead - enlil_lead_tol)
    & (_enlil_raw["lead"] <= _target_lead + enlil_lead_tol)
].copy()
_enlil_selected["lead_err"] = (_enlil_selected["lead"] - _target_lead).abs()
_enlil_selected = (
    _enlil_selected.sort_values(["time", "lead_err"])
    .drop_duplicates("time", keep="first")
    .set_index("time")
    .sort_index()
)


def _build_enlil_frame(velocity_column):
    series = pd.to_numeric(_enlil_selected[velocity_column], errors="coerce") / 1000.0
    series = series.loc[time_axis.min() : time_axis.max()]
    series = series.resample(time_freq).mean().reindex(time_axis).interpolate(method="time")
    return pd.DataFrame({"v_noaa": series})


enlil_frames = {
    "ace_earth": _build_enlil_frame("Earth_V1"),
    "stereo_a": _build_enlil_frame("STEREO_A_V1"),
}
_lead_h = _enlil_selected["lead"].dt.total_seconds() / 3600.0
print(
    f"ENLIL lead target: {enlil_lead_days:.1f} d "
    f"(+/-{enlil_lead_tol.total_seconds()/3600:.0f}h) | "
    f"selected rows: {len(_enlil_selected)} | "
    f"actual lead: {_lead_h.min():.1f}-{_lead_h.max():.1f} h (mean {_lead_h.mean():.1f}) | "
    f"ace v_noaa finite: {int(enlil_frames['ace_earth']['v_noaa'].notna().sum())} | "
    f"stereo v_noaa finite: {int(enlil_frames['stereo_a']['v_noaa'].notna().sum())}"
)


ENLIL lead target: 5.0 d (+/-12h) | selected rows: 21605 | actual lead: 108.0-120.0 h (mean 114.0) | ace v_noaa finite: 2393 | stereo v_noaa finite: 2393


## Interactive Exploration


In [ ]:
plot_sats = [
    {
        "sat": "ace_earth",
        "label": "ACE @ Earth",
        "phi_target": ballistic["earth_phi_target"],
        "r_target": ballistic["earth_r_target"],
    },
    {
        "sat": "stereo_a",
        "label": "STEREO-A",
        "phi_target": 0.0,
        "r_target": ballistic["earth_r_target"],
    },
]

satellite_frames = {
    "ace_earth": load_ace_earth_frame(),
    "stereo_a": stereo_a_df,
}
satellite_swx_frames = {"ace_earth": build_ace_earth_swx_frame(sdo_input_df)}
comparison_frames = {}
for sat_spec in plot_sats:
    sat_name = sat_spec["sat"]
    df_sat = satellite_frames[sat_name].copy()
    df_sat.attrs["label"] = sat_spec["label"]
    comparison_frames[sat_name] = build_satellite_comparison_frame(
        time_axis=time_axis,
        phi_axis=phi_axis,
        r_axis=r_axis,
        grid_raw=V_grid,
        slow_sw_pred_mask=slow_sw_pred_mask,
        df_sat=df_sat,
        df_swx=satellite_swx_frames.get(sat_name),
        df_noaa=enlil_frames.get(sat_name),
        phi_target=sat_spec["phi_target"],
        r_target=sat_spec["r_target"],
        slow_sw_speed=slow_sw_patch_speed,
        slow_sw_patch=slow_sw_patch,
        draw_slow_sw=True,
    )

for sat_spec in plot_sats:
    sat_name = sat_spec["sat"]
    df_sat = satellite_frames[sat_name]
    print(
        sat_spec["label"],
        "rows:",
        len(df_sat),
        "range:",
        df_sat.index.min(),
        "->",
        df_sat.index.max(),
    )

ACE @ Earth rows: 13234 range: 2017-01-01 07:00:00 -> 2019-01-01 00:00:00
STEREO-A rows: 2403 range: 2018-02-24 02:00:00 -> 2018-06-04 04:00:00


In [ ]:
satellite_frames["ace_earth"]["2016-12-15 16:00:00":]

,v
date,
2017-01-01 07:00:00,542.135714
2017-01-01 08:00:00,528.435833
2017-01-01 09:00:00,522.367500
2017-01-01 10:00:00,520.830000
2017-01-01 11:00:00,512.499167
...,...
2018-12-31 19:00:00,468.693220
2018-12-31 20:00:00,461.835345
2018-12-31 21:00:00,454.810000


In [ ]:
date_strs = [d.strftime("%Y-%m-%d %H:%M:%S") for d in time_axis]
print(
    f"Earth mapping check: R=215 -> r_idx={np.argmin(np.abs(r_axis - ballistic['earth_r_target']))}"
)


def plot_polar(date_str):
    plot_polar_snapshot(
        date_str=date_str,
        time_axis=time_axis,
        phi_axis=phi_axis,
        r_axis=r_axis,
        grid_raw=V_grid,
        post_vlims_raw=post_vlims_raw,
        slow_sw_pred_mask=slow_sw_pred_mask,
        slow_sw_speed=slow_sw_speed,
        comparison_frames=comparison_frames,
        draw_slow_sw=True,
    )


interact(
    plot_polar,
    date_str=widgets.SelectionSlider(
        options=date_strs,
        value=date_strs[0],
        description="date",
        continuous_update=False,
        layout=widgets.Layout(width="90%"),
    ),
)

Earth mapping check: R=215 -> r_idx=130


interactive(children=(SelectionSlider(continuous_update=False, description='date', layout=Layout(width='90%'),…

<function __main__.plot_polar(date_str)>

In [ ]:
print("swept-skip deposits:", stats.swept_skip_deposits)


swept-skip deposits: 9042913


## Forecast Scoring


Metrics are split by satellite and sample regime. ACE uses Earth ICME-catalog exclusions; STEREO-A only compares all solar wind and samples after removing predicted slow wind.


In [ ]:
SCORE_FREQ = "1h"
AU_KM = 149597870.7
SECONDS_PER_DAY = 86400.0

FORECAST_COLUMNS_BY_SAT = {
    "ace_earth": [
        ("pred", "v_predict"),
        ("prev_cr", "v_1cr_ago"),
        ("swx", "v_swx"),
        ("noaa", "v_noaa"),
    ],
    "stereo_a": [
        ("pred", "v_predict"),
        ("prev_cr", "v_1cr_ago"),
        ("noaa", "v_noaa"),
    ],
}

REGIME_ORDER = [
    "all_sw",
    "no_icme",
    "no_icme_no_slow_sw",
]
REGIME_SORT_ORDER = [
    "all_sw",
    "no_slow_sw",
    "no_icme",
    "no_icme_no_slow_sw",
]
REGIME_ORDER_BY_SAT = {
    "stereo_a": [
        "all_sw",
        "no_slow_sw",
    ],
}
SAT_LABELS = {spec["sat"]: spec["label"] for spec in plot_sats}


def build_eval_index(start_dt, end_dt, freq=SCORE_FREQ):
    return pd.date_range(pd.Timestamp(start_dt), pd.Timestamp(end_dt), freq=freq)


def prepare_eval_series(series, eval_index, output_name, freq=SCORE_FREQ):
    prepared = pd.Series(
        pd.to_numeric(series, errors="coerce").to_numpy(),
        index=pd.DatetimeIndex(series.index),
        name=output_name,
    )
    prepared = prepared[~prepared.index.duplicated(keep="last")].sort_index()
    prepared = prepared.loc[eval_index.min() : eval_index.max()]
    prepared = prepared.resample(freq).mean()
    return prepared.reindex(eval_index)


def prepare_eval_mask(series, eval_index, output_name, freq=SCORE_FREQ):
    prepared = pd.Series(
        pd.Series(series).fillna(False).astype(bool).to_numpy(),
        index=pd.DatetimeIndex(series.index),
        name=output_name,
    )
    prepared = prepared[~prepared.index.duplicated(keep="last")].sort_index()
    prepared = prepared.loc[eval_index.min() : eval_index.max()]
    prepared = prepared.resample(freq).max()
    return prepared.reindex(eval_index, fill_value=False).astype(bool)


def build_slow_sw_eval_mask(frame, eval_index):
    if "slow_sw_patch_mask" in frame.columns:
        return prepare_eval_mask(frame["slow_sw_patch_mask"], eval_index, "is_slow_sw")

    raw_column = "v_predict_raw" if "v_predict_raw" in frame.columns else "v_predict"
    raw_speed = pd.Series(
        pd.to_numeric(frame[raw_column], errors="coerce").to_numpy(),
        index=pd.DatetimeIndex(frame.index),
        name=raw_column,
    )
    slow_ref = pd.Series(
        slow_sw_speed,
        index=pd.DatetimeIndex(time_axis),
        name="slow_sw_speed",
    )
    slow_ref = (
        slow_ref.reindex(raw_speed.index).interpolate(method="time").ffill().bfill()
    )
    is_slow = pd.Series(
        np.isclose(raw_speed.to_numpy(dtype=float), slow_ref.to_numpy(dtype=float)),
        index=raw_speed.index,
        name="is_slow_sw",
    )
    return prepare_eval_mask(is_slow, eval_index, "is_slow_sw")


def build_sat_score_frame(sat_name, frame, eval_index, icme_mask):
    assert "v_real" in frame.columns, f"Missing observed speed column for {sat_name}"

    out = pd.DataFrame(index=eval_index)
    out["obs"] = prepare_eval_series(frame["v_real"], eval_index, "obs")
    out["phi_target"] = prepare_eval_series(
        frame["phi_target"], eval_index, "phi_target"
    )
    out["is_icme"] = icme_mask.reindex(eval_index, fill_value=False).astype(bool)
    out["is_slow_sw"] = build_slow_sw_eval_mask(frame, eval_index)

    for forecast_name, column in FORECAST_COLUMNS_BY_SAT[sat_name]:
        if column in frame.columns:
            out[forecast_name] = prepare_eval_series(
                frame[column], eval_index, forecast_name
            )

    return out


def build_regime_masks(eval_frame):
    return {
        "all_sw": pd.Series(True, index=eval_frame.index),
        "no_slow_sw": ~eval_frame["is_slow_sw"],
        "no_icme": ~eval_frame["is_icme"],
        "no_icme_no_slow_sw": ~eval_frame["is_icme"] & ~eval_frame["is_slow_sw"],
    }


def get_regime_order(sat_name):
    return REGIME_ORDER_BY_SAT.get(sat_name, REGIME_ORDER)


def compute_forecast_stats(actual, forecast, sample_mask):
    paired = pd.concat(
        [
            pd.to_numeric(actual, errors="coerce").rename("actual"),
            pd.to_numeric(forecast, errors="coerce").rename("forecast"),
            sample_mask.rename("sample_mask"),
        ],
        axis=1,
    )
    paired = paired.loc[paired["sample_mask"]].dropna(subset=["actual", "forecast"])
    if len(paired) == 0:
        return {
            "n_samples": 0,
            "r": np.nan,
            "rmse": np.nan,
            "mae": np.nan,
            "bias": np.nan,
        }

    err = paired["forecast"] - paired["actual"]
    r_value = np.nan
    if len(paired) >= 2:
        r_value = float(paired["actual"].corr(paired["forecast"]))

    return {
        "n_samples": int(len(paired)),
        "r": r_value,
        "rmse": float(np.sqrt(np.mean(err**2))),
        "mae": float(np.mean(np.abs(err))),
        "bias": float(np.mean(err)),
    }

In [ ]:
score_index = build_eval_index(start_dt, end_dt)
icme_eval_mask = build_icme_mask(score_index)
score_frames = {
    sat_name: build_sat_score_frame(
        sat_name=sat_name,
        frame=frame,
        eval_index=score_index,
        icme_mask=icme_eval_mask,
    )
    for sat_name, frame in comparison_frames.items()
}

score_rows = []
for sat_name, eval_frame in score_frames.items():
    sat_label = SAT_LABELS.get(sat_name, sat_name)
    regime_masks = build_regime_masks(eval_frame)
    for regime in get_regime_order(sat_name):
        sample_mask = regime_masks[regime]
        for forecast_name, _column in FORECAST_COLUMNS_BY_SAT[sat_name]:
            if forecast_name not in eval_frame.columns:
                continue
            score_rows.append(
                {
                    "sat": sat_label,
                    "regime": regime,
                    "forecast": forecast_name,
                    **compute_forecast_stats(
                        actual=eval_frame["obs"],
                        forecast=eval_frame[forecast_name],
                        sample_mask=sample_mask,
                    ),
                }
            )

forecast_skill_df = pd.DataFrame(score_rows)
forecast_skill_df["sat_order"] = forecast_skill_df["sat"].map(
    {label: order for order, label in enumerate(SAT_LABELS.values())}
)
forecast_skill_df["regime_order"] = forecast_skill_df["regime"].map(
    {regime: order for order, regime in enumerate(REGIME_SORT_ORDER)}
)
forecast_skill_df["forecast_order"] = forecast_skill_df["forecast"].map(
    {"pred": 0, "prev_cr": 1, "swx": 2, "noaa": 3}
)
forecast_skill_df = (
    forecast_skill_df.sort_values(["sat_order", "regime_order", "forecast_order"])
    .drop(columns=["sat_order", "regime_order", "forecast_order"])
    .set_index(["sat", "regime", "forecast"])
    .round({"r": 3, "rmse": 1, "mae": 1, "bias": 1})
)
forecast_skill_df

### v_predict vs WSA/ENLIL


In [ ]:
# Compare our model's v_predict against the WSA/ENLIL prediction (v_noaa),
# treating v_noaa as the reference series.
pred_vs_noaa_rows = []
for sat_name, eval_frame in score_frames.items():
    if "noaa" not in eval_frame.columns or "pred" not in eval_frame.columns:
        continue
    sat_label = SAT_LABELS.get(sat_name, sat_name)
    regime_masks = build_regime_masks(eval_frame)
    for regime in get_regime_order(sat_name):
        pred_vs_noaa_rows.append(
            {
                "sat": sat_label,
                "regime": regime,
                **compute_forecast_stats(
                    actual=eval_frame["noaa"],
                    forecast=eval_frame["pred"],
                    sample_mask=regime_masks[regime],
                ),
            }
        )

pred_vs_noaa_df = pd.DataFrame(pred_vs_noaa_rows)
pred_vs_noaa_df["sat_order"] = pred_vs_noaa_df["sat"].map(
    {label: order for order, label in enumerate(SAT_LABELS.values())}
)
pred_vs_noaa_df["regime_order"] = pred_vs_noaa_df["regime"].map(
    {regime: order for order, regime in enumerate(REGIME_SORT_ORDER)}
)
pred_vs_noaa_df = (
    pred_vs_noaa_df.sort_values(["sat_order", "regime_order"])
    .drop(columns=["sat_order", "regime_order"])
    .set_index(["sat", "regime"])
    .round({"r": 3, "rmse": 1, "mae": 1, "bias": 1})
)
pred_vs_noaa_df


In [ ]:
def compute_ch_age_days(phi_deg, speed_km_s):
    phi = np.asarray(phi_deg, dtype=float)
    speed = np.asarray(speed_km_s, dtype=float)
    age = np.full(phi.shape, np.nan, dtype=float)
    valid = np.isfinite(phi) & np.isfinite(speed) & (speed > 0.0)
    age[valid] = (
        np.mod(phi[valid], 360.0) / 360.0
    ) * CARRINGTON_ROTATION_DAYS + AU_KM / speed[valid] / SECONDS_PER_DAY
    return age


ch_age_rows = []
for sat_name, eval_frame in score_frames.items():
    sat_label = SAT_LABELS.get(sat_name, sat_name)
    age_days = pd.Series(
        compute_ch_age_days(
            phi_deg=eval_frame["phi_target"].to_numpy(dtype=float),
            speed_km_s=eval_frame["pred"].to_numpy(dtype=float),
        ),
        index=eval_frame.index,
        name="ch_age_days",
    )
    regime_masks = build_regime_masks(eval_frame)
    for regime in get_regime_order(sat_name):
        sample = age_days.loc[regime_masks[regime]].dropna()
        ch_age_rows.append(
            {
                "sat": sat_label,
                "regime": regime,
                "n_samples": int(len(sample)),
                "mean_ch_age_days": float(sample.mean()) if len(sample) else np.nan,
            }
        )

ch_age_df = pd.DataFrame(ch_age_rows)
ch_age_df["sat_order"] = ch_age_df["sat"].map(
    {label: order for order, label in enumerate(SAT_LABELS.values())}
)
ch_age_df["regime_order"] = ch_age_df["regime"].map(
    {regime: order for order, regime in enumerate(REGIME_SORT_ORDER)}
)
ch_age_df = (
    ch_age_df.sort_values(["sat_order", "regime_order"])
    .drop(columns=["sat_order", "regime_order"])
    .set_index(["sat", "regime"])
    .round({"mean_ch_age_days": 2})
)
ch_age_df

### Animation Export


In [ ]:
if True:
    anim_fps = 30
    anim_1h_mult = 2.0
    anim_dpi = runtime["animation_dpi"]
    anim_outfile = Path("Outputs/SW/SW Polar Animation.mp4")

    primary_sat = plot_sats[0]["sat"]
    primary_frame_anim = comparison_frames[primary_sat]
    animation_stats = export_polar_animation(
        anim_outfile=anim_outfile,
        time_axis=time_axis,
        phi_axis=phi_axis,
        r_axis=r_axis,
        grid_raw=V_grid,
        post_vlims_raw=post_vlims_raw,
        slow_sw_pred_mask=slow_sw_pred_mask,
        comparison_frames=comparison_frames,
        time_step_minutes=time_step_minutes,
        slow_sw_speed=slow_sw_speed,
        draw_slow_sw=True,
        anim_fps=anim_fps,
        anim_1h_mult=anim_1h_mult,
        anim_dpi=anim_dpi,
    )
    print(animation_stats)

Export MP4:   0%|          | 0/1202 [00:00<?, ?frame/s]

{'frames': 1202.0, 'stride': 2.0, 'frame_interval_minutes': 120.0, 'fps': 30.0}


# Outputs


In [ ]:
# out_dir = Path("Outputs/SW")
# out_dir.mkdir(parents=True, exist_ok=True)
# out_parquet = out_dir / "SW Earth Series.parquet"

# comparison_frames[primary_sat].loc[time_axis.min() : time_axis.max()].to_parquet(out_parquet)
# print("Saved", out_parquet)

In [ ]:
# earth_phi_target = float(ballistic["earth_phi_target"])
# earth_r_target = float(ballistic["earth_r_target"])
# phi0_idx = int(np.argmin(np.abs(phi_axis - earth_phi_target)))
# r215_idx = int(np.argmin(np.abs(r_axis - earth_r_target)))

# propagated_earth = pd.Series(
#     V_grid[:, phi0_idx, r215_idx],
#     index=time_axis,
#     name=f"v_propagated_r{int(earth_r_target)}_phi{int(earth_phi_target)}",
# )


# def classify_gap_transition(left_speed, right_speed):
#     if not np.isfinite(left_speed) or not np.isfinite(right_speed):
#         return "edge-missing"
#     if right_speed > left_speed:
#         return "lower -> higher"
#     if right_speed < left_speed:
#         return "higher -> lower"
#     return "equal -> equal"


# gap_mask = propagated_earth.isna()
# gap_groups = gap_mask.ne(gap_mask.shift().fillna(False)).cumsum()
# gap_rows = []

# for _, gap in propagated_earth[gap_mask].groupby(gap_groups[gap_mask]):
#     gap_start = gap.index[0]
#     gap_end = gap.index[-1]
#     start_i = propagated_earth.index.get_loc(gap_start)
#     end_i = propagated_earth.index.get_loc(gap_end)

#     left_i = start_i - 1 if start_i > 0 else None
#     right_i = end_i + 1 if end_i + 1 < len(propagated_earth) else None

#     left_time = propagated_earth.index[left_i] if left_i is not None else pd.NaT
#     right_time = propagated_earth.index[right_i] if right_i is not None else pd.NaT
#     left_speed = propagated_earth.iloc[left_i] if left_i is not None else np.nan
#     right_speed = propagated_earth.iloc[right_i] if right_i is not None else np.nan
#     transition = classify_gap_transition(left_speed, right_speed)

#     gap_rows.append(
#         {
#             "gap_start": gap_start,
#             "gap_end": gap_end,
#             "gap_steps": int(len(gap)),
#             "gap_hours": float(len(gap) * time_step_hours),
#             "left_time": left_time,
#             "left_speed": left_speed,
#             "right_time": right_time,
#             "right_speed": right_speed,
#             "speed_delta": right_speed - left_speed,
#             "transition": transition,
#         }
#     )

# gap_details = pd.DataFrame(gap_rows)

# if gap_details.empty:
#     print("No propagated gaps found at (R=215, phi=0).")
# else:
#     gap_summary = (
#         gap_details.groupby("transition", dropna=False)
#         .agg(
#             gap_count=("transition", "size"),
#             total_gap_hours=("gap_hours", "sum"),
#             median_gap_hours=("gap_hours", "median"),
#         )
#         .sort_values(["gap_count", "total_gap_hours"], ascending=False)
#     )

#     print(
#         f"Propagated gaps at (R={int(earth_r_target)}, phi={int(earth_phi_target)}):",
#         len(gap_details),
#     )
#     print(gap_summary)
#     gap_details = gap_details.sort_values("gap_start").reset_index(drop=True)
#     gap_details

In [ ]:
def _find_phi_index(phi_axis, phi_target):
    delta = np.abs((np.asarray(phi_axis, dtype=float) - float(phi_target) + 180.0) % 360.0 - 180.0)
    return int(np.argmin(delta))

def _find_axis_index(axis_values, target):
    return int(np.argmin(np.abs(axis_values - float(target))))

frame = comparison_frames["stereo_a"]
time_idx_in_axis = pd.DatetimeIndex(time_axis)

# Only sample at rows that exist in time_axis (skip the +cr_days outer-joined rows)
mask = frame.index.isin(time_idx_in_axis)
sub = frame.loc[mask]
pi = np.array([_find_phi_index(phi_axis, p) for p in sub["phi_target"]])
ri = np.array([_find_axis_index(r_axis, target=r) for r in sub["r_target"]])
ti = np.searchsorted(time_idx_in_axis, sub.index)

raw_along_traj = V_grid[ti, pi, ri]
print(f"grid_raw finite along STEREO trajectory: {np.isfinite(raw_along_traj).sum()}/{len(sub)}")
print(f"v_predict finite in comparison_frame:    {sub['v_predict'].notna().sum()}/{len(sub)}")

finite_idx = np.where(np.isfinite(raw_along_traj))[0]
if len(finite_idx):
    print(f"grid_raw last finite tv at STEREO: {sub.index[finite_idx[-1]]}")
print(f"sim_start + cr_days:               {time_axis[0] + pd.Timedelta(days=27)}")
print(f"sim_end:                           {time_axis[-1]}")

# Also: does phi_target change along the way, or is it stuck (HGE-fixed vs Carrington-rotating)?
print(f"phi_target distinct values:        {sub['phi_target'].round(1).nunique()}")
print(f"r_target distinct values:          {sub['r_target'].round(1).nunique()}")
print(f"unique (phi_idx, r_idx) cells:     {len({(int(p), int(r)) for p, r in zip(pi, ri)})}")

grid_raw finite along STEREO trajectory: 1259/2403
v_predict finite in comparison_frame:    1259/2403
grid_raw last finite tv at STEREO: 2018-06-04 04:00:00
sim_start + cr_days:               2018-03-23 02:00:00
sim_end:                           2018-06-04 04:00:00
phi_target distinct values:        52
r_target distinct values:          13
unique (phi_idx, r_idx) cells:     6


In [ ]:
for cell in sorted({(int(p), int(r)) for p, r in zip(pi, ri)}):
    p, r = cell
    in_cell = (pi == p) & (ri == r)
    finite_in_cell = np.isfinite(raw_along_traj[in_cell])
    visit_first = sub.index[in_cell][0]
    visit_last  = sub.index[in_cell][-1]
    first_finite = sub.index[in_cell][np.where(finite_in_cell)[0][0]] if finite_in_cell.any() else None
    fill_threshold = time_axis[0] + pd.Timedelta(hours=float(phi_delay_offsets[p]))   # plus travel
    print(f"pv={p:2d} r_idx={r:3d}  visits={in_cell.sum():5d}  "
          f"finite={finite_in_cell.sum():5d}  first_visit={visit_first}  "
          f"first_finite={first_finite}  φ_delay_thresh={fill_threshold}")

pv=218 r_idx=125  visits=  162  finite=    0  first_visit=2018-02-24 02:00:00  first_finite=None  φ_delay_thresh=2018-03-14 06:00:00
pv=219 r_idx=125  visits=  806  finite=  279  first_visit=2018-03-02 20:00:00  first_finite=2018-03-19 09:00:00  φ_delay_thresh=2018-03-14 08:00:00
pv=220 r_idx=125  visits=  545  finite=  372  first_visit=2018-04-05 10:00:00  first_finite=2018-04-05 10:00:00  φ_delay_thresh=2018-03-14 10:00:00
pv=221 r_idx=125  visits=  402  finite=  274  first_visit=2018-04-28 03:00:00  first_finite=2018-04-28 03:00:00  φ_delay_thresh=2018-03-14 12:00:00
pv=222 r_idx=125  visits=  335  finite=  245  first_visit=2018-05-14 21:00:00  first_finite=2018-05-15 18:00:00  φ_delay_thresh=2018-03-14 14:00:00
pv=223 r_idx=125  visits=  153  finite=   89  first_visit=2018-05-28 20:00:00  first_finite=2018-05-28 20:00:00  φ_delay_thresh=2018-03-14 16:00:00


In [ ]:
assert "comparison_frames" in globals(), "Run the comparison_frames cell before this plot."

sw_plot_out = export_solar_wind_plot(
    plot_outfile="./SW.pdf",
    comparison_frames=comparison_frames,
    start_dt=start_dt,
    end_dt=end_dt,
    sat_labels=SAT_LABELS,
)
sw_plot_out
